In [ ]:
# Change the branch to yours
# And choose an L4 GPU in Runtime > Change runtime type
BRANCH = "test-khalit-clean"
REPO_URL = "git@github.com:St1p42/sglang.git"
REPO_DIR = "/content/sglang"
MODEL = "Qwen/Qwen2.5-7B-Instruct"
WORKLOAD_PREFIX = "workload_msmarco"
NUM_GROUPS = 64
GROUP_SIZE = 32
WORKLOAD_FILE = f"workloads/{WORKLOAD_PREFIX}_interleaved_{NUM_GROUPS}x{GROUP_SIZE}.jsonl"
LATENCY_REQUESTS = 128
THROUGHPUT_REQUESTS = 512
LATENCY_PARALLEL = 1
THROUGHPUT_PARALLEL = 32


In [ ]:
# Generate a public key for SSH - add the printed result to your GitHub account
%%bash
set -e

mkdir -p /root/.ssh
chmod 700 /root/.ssh

if [ ! -f /root/.ssh/id_ed25519 ]; then
  ssh-keygen -t ed25519 -C "colab" -f /root/.ssh/id_ed25519 -N ""
fi

ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null
chmod 600 /root/.ssh/known_hosts

echo "Public key:"
cat /root/.ssh/id_ed25519.pub


In [ ]:
# SSH test
%%bash
set -e
ssh -T git@github.com || true


In [ ]:
import os
for key in ["BRANCH", "REPO_URL", "REPO_DIR", "MODEL", "WORKLOAD_FILE"]:
    os.environ[key] = globals()[key]
os.environ["NUM_GROUPS"] = str(NUM_GROUPS)
os.environ["GROUP_SIZE"] = str(GROUP_SIZE)
os.environ["LATENCY_REQUESTS"] = str(LATENCY_REQUESTS)
os.environ["THROUGHPUT_REQUESTS"] = str(THROUGHPUT_REQUESTS)
os.environ["LATENCY_PARALLEL"] = str(LATENCY_PARALLEL)
os.environ["THROUGHPUT_PARALLEL"] = str(THROUGHPUT_PARALLEL)


In [ ]:
# Clone and checkout the branch indicated on the top
# After running, you should be able to see the repo in the Files in the left menu bar
%%bash
set -e

if [ ! -d "$REPO_DIR/.git" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
git fetch origin
git checkout "$BRANCH"
git pull origin "$BRANCH"
git branch --show-current
git log --oneline -1


In [ ]:
# See GPU/CPU/RAM info
%%bash
echo "=== GPU Info ==="
nvidia-smi || echo "No GPU attached or nvidia-smi failed"
echo -e "\n=== CPU Info ==="
lscpu | head -n 20
echo -e "\n=== Memory Info ==="
free -h


In [ ]:
# Install
%%bash
set -e
cd /content/sglang

python -m pip uninstall -y sglang sgl-kernel flashinfer-python vllm || true
python -m pip install -e python
python -m pip install -q pandas numpy requests tqdm aiohttp dspy-ai rouge-score datasets


In [ ]:
# You should see the local SGLang package
import sys
sys.path = [p for p in sys.path if p != "/content/sglang"]
sys.path.insert(0, "/content/sglang/python")
import sglang, torch
from importlib.metadata import version
print("torch:", torch.__version__)
print("sglang version:", version("sglang"))
print("sglang path:", list(sglang.__path__))
print("sglang spec origin:", sglang.__spec__.origin)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))


In [ ]:
# Generate the MSMARCO-derived workload once
%%bash
set -e
cd /content/sglang/690AB/eval/experiment_1/rag_dspy_style
mkdir -p workloads
python generate_msmarco_cache_workload.py --num-groups "$NUM_GROUPS" --group-size "$GROUP_SIZE" --out-dir workloads
ls -lh workloads


In [ ]:
# Stop the SGLang server when needed
!pkill -f "sglang.launch_server" || true


In [ ]:
# Verify SGLang is gone
!ps aux | grep "sglang.launch_server" | grep -v grep


In [ ]:
# Combined cleanup before any new server launch
%%bash
pkill -f "sglang.launch_server" || true
sleep 2
ps aux | grep "sglang.launch_server" | grep -v grep || true


In [ ]:
# Launch the SGLang server with custom radix-cache-impl + custom cache-aware scheduling (HiCache off)
%%bash
cd /content/sglang
nohup python -m sglang.launch_server \
  --model-path "$MODEL" \
  --port 30000 \
  --radix-cache-impl custom \
  --cache-aware-scheduling custom \
  > /content/sglang_server.log 2>&1 &

echo $! > /content/sglang_server.pid
sleep 20
tail -n 50 /content/sglang_server.log


In [ ]:
# Check SGLang readiness
%%bash
curl -s http://127.0.0.1:30000/v1/models || true
echo
tail -n 100 /content/sglang_server.log


In [ ]:
# Run the DSPy-style RAG benchmark against SGLang once
# Warmup, flush cache, then run the real benchmark
%%bash
set -e
cd /content/sglang/690AB/eval/experiment_1/rag_dspy_style
mkdir -p results
rm -f results/sglang_single.jsonl results/sglang_single_raw.json
TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py --backend sglang --model "$MODEL" --dataset-file "$WORKLOAD_FILE" --max-requests 8 --parallel 1 --result-file /tmp/sglang_rag_warmup.jsonl > /tmp/bench_rag_sglang_warmup.log
curl -s -X POST http://127.0.0.1:30000/flush_cache
sleep 2
TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py --backend sglang --model "$MODEL" --dataset-file "$WORKLOAD_FILE" --max-requests "$LATENCY_REQUESTS" --parallel "$LATENCY_PARALLEL" --result-file results/sglang_single.jsonl --raw-result-file results/sglang_single_raw.json --run-id single


In [ ]:
# Inspect the saved single-run SGLang result
%%bash
cd /content/sglang/690AB/eval/experiment_1/rag_dspy_style
tail -n 1 results/sglang_single.jsonl


In [ ]:
# Run the latency-style DSPy RAG benchmark 5 times against SGLang
%%bash
set -e
cd /content/sglang/690AB/eval/experiment_1/rag_dspy_style
mkdir -p results
rm -f results/sglang_latency.jsonl results/sglang_latency_run*.json
TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py --backend sglang --model "$MODEL" --dataset-file "$WORKLOAD_FILE" --max-requests 8 --parallel 1 --result-file /tmp/sglang_rag_latency_warmup.jsonl > /tmp/bench_rag_sglang_latency_warmup.log
for run_id in 1 2 3 4 5; do
  curl -s -X POST http://127.0.0.1:30000/flush_cache
  sleep 2
  TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py --backend sglang --model "$MODEL" --dataset-file "$WORKLOAD_FILE" --max-requests "$LATENCY_REQUESTS" --parallel "$LATENCY_PARALLEL" --result-file results/sglang_latency.jsonl --raw-result-file results/sglang_latency_run${run_id}.json --run-id ${run_id}
done


In [ ]:
# Aggregate the latency-style SGLang runs
import glob, json, pandas as pd

rows = []
for path in sorted(glob.glob("/content/sglang/690AB/eval/experiment_1/rag_dspy_style/results/sglang_latency_run*.json")):
    with open(path) as f:
        obj = json.load(f)
    rows.append({
        "run_id": obj.get("run_id"),
        "mean_e2e_latency_ms": obj["metrics"]["mean_e2e_latency_ms"],
        "mean_ttft_ms": obj["metrics"]["mean_ttft_ms"],
        "p95_e2e_latency_ms": obj["extended_metrics"].get("p95_e2e_latency_ms"),
        "request_throughput": obj["metrics"]["request_throughput"],
        "cache_hit_rate": obj.get("cache_hit_rate"),
        "rouge_l": obj.get("answer_quality", {}).get("rouge_l"),
        "token_f1": obj.get("answer_quality", {}).get("token_f1"),
    })
df = pd.DataFrame(rows)
display(df)
summary = df.drop(columns=["run_id"]).agg(["mean", "std"]).T.reset_index().rename(columns={"index": "metric"})
display(summary)


In [ ]:
# Run the throughput-style DSPy RAG benchmark 5 times against SGLang
%%bash
set -e
cd /content/sglang/690AB/eval/experiment_1/rag_dspy_style
mkdir -p results
rm -f results/sglang_throughput.jsonl results/sglang_throughput_run*.json
TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py --backend sglang --model "$MODEL" --dataset-file "$WORKLOAD_FILE" --max-requests 8 --parallel 1 --result-file /tmp/sglang_rag_throughput_warmup.jsonl > /tmp/bench_rag_sglang_throughput_warmup.log
for run_id in 1 2 3 4 5; do
  curl -s -X POST http://127.0.0.1:30000/flush_cache
  sleep 2
  TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py --backend sglang --model "$MODEL" --dataset-file "$WORKLOAD_FILE" --max-requests "$THROUGHPUT_REQUESTS" --parallel "$THROUGHPUT_PARALLEL" --result-file results/sglang_throughput.jsonl --raw-result-file results/sglang_throughput_run${run_id}.json --run-id ${run_id}
done


In [ ]:
# Aggregate the throughput-style SGLang runs
import glob, json, pandas as pd

rows = []
for path in sorted(glob.glob("/content/sglang/690AB/eval/experiment_1/rag_dspy_style/results/sglang_throughput_run*.json")):
    with open(path) as f:
        obj = json.load(f)
    rows.append({
        "run_id": obj.get("run_id"),
        "mean_e2e_latency_ms": obj["metrics"]["mean_e2e_latency_ms"],
        "mean_ttft_ms": obj["metrics"]["mean_ttft_ms"],
        "p95_e2e_latency_ms": obj["extended_metrics"].get("p95_e2e_latency_ms"),
        "request_throughput": obj["metrics"]["request_throughput"],
        "cache_hit_rate": obj.get("cache_hit_rate"),
        "rouge_l": obj.get("answer_quality", {}).get("rouge_l"),
        "token_f1": obj.get("answer_quality", {}).get("token_f1"),
    })
df = pd.DataFrame(rows)
display(df)
summary = df.drop(columns=["run_id"]).agg(["mean", "std"]).T.reset_index().rename(columns={"index": "metric"})
display(summary)
